# Natural Language Processing

Natural language processing (NLP) involves using computers to process and analyze human language. To work with text, we represent it numerically.


## Preparing the text

We begin by extracting text from a PDF, then examine ways to split it into tokens and represent those tokens numerically.


## Extracting text from a PDF

We use PyPDF2 to extract the text of a book named `2038` from a PDF. The extracted text becomes the input for the tokenization and representation examples that follow.

In [5]:
from pathlib import Path
import sys

import pypdf

sys.path.insert(0, str(Path.cwd().parent))


pdf_path = "/home/nick/github-projects/prompting-base-models/data/2038.pdf"


text = ""
for page in pypdf.PdfReader(pdf_path).pages:
    text += page.extract_text()




print(f"Extracted {len(text):,} characters.")
print("\nText preview:\n")
print(text[10:150])

Extracted 233,682 characters.

Text preview:

 K K AT S A R E L A SCHAPTER  ONE
At ﬁve o’clock, the screen in front of Winston Reed presented
its ﬁnal two of the day.
Two corners. In the


## Tokens

Tokenization splits text into smaller units called tokens. Depending on the tokenizer, a token can be a word, part of a word, punctuation mark, number, or symbol.

In [6]:
tokens = text.split()

print(f"Number of whitespace-separated tokens: {len(tokens):,}")
print("\nFirst 10 tokens:")
print(tokens[:10])

Number of whitespace-separated tokens: 45,556

First 10 tokens:
['2038', 'N', 'I', 'C', 'K', 'K', 'AT', 'S', 'A', 'R']


## Tokenizing with NLTK

Whitespace splitting leaves punctuation attached to words. We now use NLTK’s word tokenizer to separate words and punctuation into tokens.

In [8]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab", quiet=True)

nltk_tokens = word_tokenize(text)

print(f"Number of NLTK tokens: {len(nltk_tokens):,}")
print("\nFirst 10 tokens:")
print(nltk_tokens[10:20])

Number of NLTK tokens: 57,160

First 10 tokens:
['E', 'L', 'A', 'SCHAPTER', 'ONE', 'At', 'ﬁve', 'o', '’', 'clock']


## Recognizing names and places

Tokenization splits text into pieces. Named entity recognition adds labels to spans such as people, places, and organizations. Here, we use spaCy to find these in a short excerpt from *1984*.

In [11]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(text[:2000])

print("Named entities:\n")

for entity in doc.ents:
    print(
        f"{entity.text:<30} "
        f"{entity.label_:<12} "
        f"{spacy.explain(entity.label_)}"
    )

Named entities:

2038                           DATE         Absolute or relative dates or periods
ONE                            CARDINAL     Numerals that do not fall under another type
Winston Reed                   ORG          Companies, agencies, institutions, etc.
two                            CARDINAL     Numerals that do not fall under another type
Two                            CARDINAL     Numerals that do not fall under another type
second                         ORDINAL      "first", "second", etc.
Underneath                     PERSON       People, including fictional
Reed                           PERSON       People, including fictional
second                         ORDINAL      "first", "second", etc.
eleven years                   DATE         Absolute or relative dates or periods
a few hundred                  CARDINAL     Numerals that do not fall under another type
Reed                           PERSON       People, including fictional
1as                        

### About the output

Each row shows a span of text, its predicted entity label, and the label’s meaning. For example, `PERSON` identifies a person’s name, while `GPE` identifies a country, city, or state.

These are predictions based on context. Fictional names and places may be missed or assigned an incorrect label.

## Converting tokens into numbers

We can assign each unique token an integer ID. Repeated occurrences of the same token receive the same ID.

### Label encoding

Here, we use `LabelEncoder` to demonstrate this mapping. The IDs identify tokens; their numerical values do not represent meaning, similarity, or importance.

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
token_ids = le.fit_transform(nltk_tokens)

print(f"Number of tokens: {len(token_ids):,}")
print(f"Number of unique tokens: {len(le.classes_):,}")
print("\nFirst 20-50 tokens and their IDs:")

for token, token_id in zip(nltk_tokens[20:50], token_ids[20:50]):
    print(f"{token:<20} {token_id}")

Number of tokens: 57,160
Number of unique tokens: 3,201

First 20-50 tokens and their IDs:
,                    1
the                  2814
screen               2449
in                   1677
front                1431
of                   2073
Winston              611
Reed                 478
presented            2259
its                  1703
ﬁnal                 3171
two                  2945
of                   2073
the                  2814
day                  1133
.                    2
Two                  582
corners              1078
.                    2
In                   356
the                  2814
ﬁrst                 3182
,                    1
the                  2814
ball                 768
was                  3030
swung                2763
in                   1677
hard                 1565
and                  685


### About the output

Each token is paired with its integer ID. Repeated tokens receive the same ID, while different tokens receive different IDs.

These IDs are labels, not measurements: nearby numbers do not imply similar meanings. The same token also keeps its ID across different contexts, so this representation cannot distinguish its different uses.

### Same token, different meanings

“Orange” can refer to a fruit or a color:

- “She peeled an orange for breakfast.”
- “He painted the door orange.”

The code below highlights both occurrences and displays their token IDs. Label encoding assigns `orange` the same ID in both sentences, even though its meaning changes with context.

In [16]:
from IPython.display import HTML, display

sentences = [
    "She peeled an orange for breakfast.",
    "He painted the door orange.",
]

sentence_tokens = [word_tokenize(sentence) for sentence in sentences]

example_encoder = LabelEncoder()
example_encoder.fit([token for tokens in sentence_tokens for token in tokens])

for sentence, tokens in zip(sentences, sentence_tokens):
    token_ids = example_encoder.transform(tokens)
    orange_id = token_ids[tokens.index("orange")]

    highlighted = sentence.replace(
        "orange",
        '<strong style="color: #e67e22;">orange</strong>',
    )
    display(HTML(f"<p>{highlighted}<br>Token ID: <b>{orange_id}</b></p>"))

### About the output

Both highlighted occurrences of `orange` have the same token ID. In the first sentence, it refers to a fruit; in the second, a color.

The ID identifies the token but does not represent its meaning in context. Later, contextual embeddings will let us represent the same word differently depending on the surrounding text.

## TF-IDF

Token IDs identify words but do not measure their importance in a document. TF-IDF assigns weights based on how often a term appears within a document and how common it is across the collection.

Terms that occur frequently in one document but rarely in others tend to receive higher weights. This helps distinguish documents by their vocabulary, but it does not resolve meanings such as “orange” as a fruit versus a color.


In [18]:
import pandas as pd
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

documents = sent_tokenize(text)[2:10]

tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(documents)

for i, sentence in enumerate(documents, 1):
    print(f"{i}: {sentence}\n")

pd.DataFrame(
    X.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=[f"Sentence {i}" for i in range(1, len(documents) + 1)],
).round(3)

1: In the ﬁrst, the ball was swung in hard and
low and met by a defender who cleared it into the stands.

2: In
the second, the ball was swung in hard and low and met by
nobody at all, and drifted out for a goal kick while the striker
looked at the sky as though it had personally let him down.

3: Underneath was the usual question: Which do you prefer?

4: Reed preferred the second.

5: It was worse football but better
telly.

6: He didn’t know how the transistors got from his choice
to the match on the box, and had never once felt the need to
ﬁnd out; he only knew that whatever he picked went into it
somewhere, the way a brick goes into a wall, and that was
good enough for him.

7: He selected it, and the screen went quiet.

8: It did not thank him.



,ball,better,box,brick,choice,cleared,defender,did,didn,drifted,...,thank,transistors,underneath,usual,wall,way,went,worse,ﬁnd,ﬁrst
Sentence 1,0.306,0.0,0.000,0.000,0.000,0.365,0.365,0.000,0.000,0.000,...,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.000,0.365
Sentence 2,0.240,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.286,...,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.000,0.000
Sentence 3,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.5,0.5,0.000,0.000,0.000,0.0,0.000,0.000
Sentence 4,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.000,0.000
Sentence 5,0.000,0.5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.5,0.000,0.000
Sentence 6,0.000,0.0,0.238,0.238,0.238,0.000,0.000,0.000,0.238,0.000,...,0.000,0.238,0.0,0.0,0.238,0.238,0.199,0.0,0.238,0.000
Sentence 7,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.0,0.0,0.000,0.000,0.436,0.0,0.000,0.000
Sentence 8,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.707,0.000,0.000,...,0.707,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.000,0.000


### About the output

Each row represents a sentence from *1984*, and each column represents a term. The values are TF-IDF weights: higher values indicate terms that help distinguish that sentence from the others.

A zero means the term does not appear in that sentence. Common English stop words are excluded. These weights describe vocabulary patterns, not the meaning of a word in context.

## Dense token embeddings

An embedding represents a token as a vector of numbers. In trained models, these vectors can capture useful relationships between tokens.

We begin with randomly initialized vectors to show the structure of an embedding matrix. These vectors have not been trained, so their distances do not yet represent meaningful similarities.

A fixed embedding assigns each token the same vector wherever it appears. Contextual embeddings, such as those produced by BERT, can represent the same token differently depending on its surrounding text.

In [19]:
import numpy as np
import pandas as pd

sample_tokens = nltk_tokens[:10]
unique_tokens = list(dict.fromkeys(sample_tokens))

embedding_dim = 4
rng = np.random.default_rng(42)
embedding_matrix = rng.normal(
    size=(len(unique_tokens), embedding_dim)
)

pd.DataFrame(
    embedding_matrix,
    index=unique_tokens,
    columns=[f"Dimension {i}" for i in range(1, embedding_dim + 1)],
).round(3)

,Dimension 1,Dimension 2,Dimension 3,Dimension 4
2038,0.305,-1.040,0.750,0.941
N,-1.951,-1.302,0.128,-0.316
I,-0.017,-0.853,0.879,0.778
C,0.066,1.127,0.468,-0.859
K,0.369,-0.959,0.878,-0.050
AT,-0.185,-0.681,1.223,-0.155
S,-0.428,-0.352,0.532,0.365
A,0.413,0.431,2.142,-0.406
R,-0.512,-0.814,0.616,1.129


### About the output

Each row represents a unique token from the first 10 NLTK tokens, and each column is one dimension of its vector.

The values are randomly initialized, so nearby vectors do not indicate similar meanings. This demonstrates the structure of an embedding matrix; learning useful values requires training.